In [ ]:
%pip install -U -q transformers datasets accelerate scikit-learn "protobuf==3.20.3" sentencepiece

In [ ]:
import numpy as np
import torch
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback
)
import warnings
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = 'cuda'

In [ ]:
ds = load_dataset("ailsntua/QEvasion")

train_full_df = ds['train'].to_pandas()
test_df = ds['test'].to_pandas()

train_df, val_df = train_test_split(
    train_full_df,
    test_size=0.2,
    random_state=SEED,
    stratify=train_full_df['clarity_label']
)

In [ ]:
MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

label2id = {"Clear Reply": 0, "Ambivalent": 1, "Clear Non-Reply": 2}
id2label = {v: k for k, v in label2id.items()}
clarity_labels = ["Clear Reply", "Ambivalent", "Clear Non-Reply"]

In [ ]:
def tokenize_function(examples):
    questions = [f"Question: {q}" for q in examples["question"]]
    answers = [f"Answer: {a}" for a in examples["interview_answer"]]
    
    combined = [f"{q}\n{a}" for q, a in zip(questions, answers)]
    
    tokenized = tokenizer(
        combined,
        truncation=False, 
        padding=False,
        max_length=None,
    )
    tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    
    return tokenized

train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

train_tokenized = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
val_tokenized = val_dataset.map(tokenize_function, batched=True, remove_columns=val_dataset.column_names)

from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class HierarchicalDataCollator:
    tokenizer: Any
    
    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        
        features_clean = [{k: v for k, v in f.items() if k != "labels"} for f in features]
        
        batch = self.tokenizer.pad(
            features_clean,
            padding=True,
            return_tensors="pt"
        )
        
        batch["labels"] = labels
        return batch

data_collator = HierarchicalDataCollator(tokenizer=tokenizer)

In [ ]:
import torch.nn as nn
from transformers import DebertaV2Model, DebertaV2PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
from typing import Optional, Tuple

class DeBERTaHierarchicalMeanPooling(DebertaV2PreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.num_labels = config.num_labels
        self.config = config
        
        self.deberta = DebertaV2Model(config)
        
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        
        self.post_init()
    
    def create_chunks_batched(self, input_ids, attention_mask, max_length=512, stride=256):
        batch_size, seq_len = input_ids.shape
        
        all_chunk_ids = []
        all_chunk_masks = []
        chunk_to_sample = []  
        
        for sample_idx in range(batch_size):
            sample_input_ids = input_ids[sample_idx]
            sample_attention_mask = attention_mask[sample_idx]
            
            actual_length = sample_attention_mask.sum().item()
            
            if actual_length <= max_length:
                chunk_ids = sample_input_ids[:max_length]
                chunk_mask = sample_attention_mask[:max_length]
                
                if len(chunk_ids) < max_length:
                    padding_length = max_length - len(chunk_ids)
                    chunk_ids = torch.cat([
                        chunk_ids,
                        torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                    ])
                    chunk_mask = torch.cat([
                        chunk_mask,
                        torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                    ])
                
                all_chunk_ids.append(chunk_ids)
                all_chunk_masks.append(chunk_mask)
                chunk_to_sample.append(sample_idx)
            else:
                start = 0
                while start < actual_length:
                    end = min(start + max_length, actual_length)
                    
                    chunk_ids = sample_input_ids[start:end]
                    chunk_mask = sample_attention_mask[start:end]
                    
                    if len(chunk_ids) < max_length:
                        padding_length = max_length - len(chunk_ids)
                        chunk_ids = torch.cat([
                            chunk_ids,
                            torch.zeros(padding_length, dtype=torch.long, device=input_ids.device)
                        ])
                        chunk_mask = torch.cat([
                            chunk_mask,
                            torch.zeros(padding_length, dtype=torch.long, device=attention_mask.device)
                        ])
                    
                    all_chunk_ids.append(chunk_ids)
                    all_chunk_masks.append(chunk_mask)
                    chunk_to_sample.append(sample_idx)
                    
                    start += stride
                    if end >= actual_length:
                        break
        
        all_chunk_ids = torch.stack(all_chunk_ids, dim=0)
        all_chunk_masks = torch.stack(all_chunk_masks, dim=0)
        chunk_to_sample = torch.tensor(chunk_to_sample, dtype=torch.long, device=input_ids.device)
        
        return all_chunk_ids, all_chunk_masks, chunk_to_sample
    
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None,
        output_attentions=None,
        output_hidden_states=None,
        return_dict=None,
    ):
        return_dict = return_dict if return_dict is not None else self.config.use_return_dict
        
        batch_size = input_ids.shape[0]

        all_chunk_ids, all_chunk_masks, chunk_to_sample = self.create_chunks_batched(
            input_ids, attention_mask, max_length=512, stride=256
        )

        outputs = self.deberta(
            input_ids=all_chunk_ids,
            attention_mask=all_chunk_masks,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=True,
        )

        all_cls_embeddings = outputs.last_hidden_state[:, 0, :]

        batch_embeddings = []
        for sample_idx in range(batch_size):

            sample_mask = chunk_to_sample == sample_idx
            sample_chunk_embeddings = all_cls_embeddings[sample_mask]  
            
         
            pooled_embedding = torch.max(sample_chunk_embeddings, dim=0)  
            batch_embeddings.append(pooled_embedding)
        

        pooled_output = torch.stack(batch_embeddings, dim=0)
        
        # classification
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))
        
        if not return_dict:
            output = (logits,)
            return ((loss,) + output) if loss is not None else output
        
        return SequenceClassifierOutput(
            loss=loss,
            logits=logits,
            hidden_states=None,
            attentions=None,
        )

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='macro')
    return {'accuracy': accuracy, 'f1': f1}

In [ ]:
class EarlyStoppingCallback(TrainerCallback):
    def __init__(self, patience: int = 3, metric_name: str = "eval_f1", greater_is_better: bool = True):
        self.patience = patience
        self.metric_name = metric_name
        self.greater_is_better = greater_is_better
        self.best_metric = None
        self.patience_counter = 0
        
    def on_evaluate(self, args, state, control, metrics, **kwargs):
        current_metric = metrics.get(self.metric_name)
        
        if current_metric is None:
            return
        
        if self.best_metric is None:
            self.best_metric = current_metric
            self.patience_counter = 0
        else:
            if self.greater_is_better:
                improved = current_metric > self.best_metric
            else:
                improved = current_metric < self.best_metric
            
            if improved:
                self.best_metric = current_metric
                self.patience_counter = 0
            else:
                self.patience_counter += 1
                
        if self.patience_counter >= self.patience:
            print(f"\nEarly stopping triggered!")
            print(f"Best {self.metric_name}: {self.best_metric:.4f}")
            control.should_training_stop = True

In [ ]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

model = DeBERTaHierarchicalMeanPooling(config)

from transformers import AutoModel
base_model = AutoModel.from_pretrained(MODEL_NAME)
model.deberta.load_state_dict(base_model.state_dict())

In [ ]:
training_args = TrainingArguments(
    output_dir="./results_deberta_hierarchical",
    learning_rate=2e-6,
    per_device_train_batch_size=8,  
    per_device_eval_batch_size=8,
    num_train_epochs=15,
    warmup_ratio=0.15,
    weight_decay=0.01,
    max_grad_norm=1.0,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    bf16=True,
    bf16_full_eval=True,
    optim="adamw_torch",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    logging_strategy="steps",
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(patience=3, metric_name="eval_f1", greater_is_better=True)],
)

In [ ]:
trainer.train()

In [ ]:
def tokenize_test(examples):
    questions = [f"Question: {q}" for q in examples["question"]]
    answers = [f"Answer: {a}" for a in examples["interview_answer"]]
    combined = [f"{q}\n{a}" for q, a in zip(questions, answers)]

    tokenized = tokenizer(
        combined,
        truncation=False,
        padding=False,
        max_length=None,
    )
    tokenized["labels"] = [label2id[l] for l in examples["clarity_label"]]
    
    return tokenized

test_dataset = Dataset.from_pandas(test_df, preserve_index=False)
test_tokenized = test_dataset.map(tokenize_test, batched=True, remove_columns=test_dataset.column_names)

predictions_output = trainer.predict(test_tokenized)
predictions = np.argmax(predictions_output.predictions, axis=-1)
predicted_labels = [id2label[pred] for pred in predictions]

y_true = test_df['clarity_label'].tolist()
y_pred = predicted_labels

accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro', labels=clarity_labels, zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', labels=clarity_labels, zero_division=0)
per_class_f1 = f1_score(y_true, y_pred, average=None, labels=clarity_labels, zero_division=0)

print(f"\nAccuracy: {accuracy:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print(f"Weighted F1: {weighted_f1:.4f}")

print(f"\nPer-class F1 scores:")
for label, f1_val in zip(clarity_labels, per_class_f1):
    print(f"  {label}: {f1_val:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=clarity_labels, zero_division=0))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred, labels=clarity_labels)
print(f"Labels: {clarity_labels}")
print(cm)